# Dollar Neutral: TXN Long / KVUE Short (1:1.135 Hedge Ratio)

This notebook demonstrates the `DollarNeutral` strategy using Texas Instruments (TXN) as the long leg
and Kenvue (KVUE) as the short leg, with a **1:1.135 hedge ratio** (long \$1 TXN, short \$1.135 KVUE).

BIL (SPDR Bloomberg 1-3 Month T-Bill ETF) absorbs collateral and residual cash.

**Data**: Yahoo Finance (auto-adjusted)  
**Rebalance**: Monthly mid-month (`month_mid` = 15th of each month)  
**Tolerance**: 5% imbalance before forced rebalance


In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio.helpers.data import YFinance
from tiportfolio import (
    DollarNeutral, FixRatio, Schedule, ScheduleBasedEngine,
    compare_strategies, plot_strategy_comparison_interactive,
    rebalance_decisions_table,
)

enable_data_source_cache("tiportfolio", cache_dir=".cache")

LONG   = "TXN"
SHORT  = "KVUE"
CASH   = "BIL"
RATIO  = 1.135           # long $1 TXN, short $1.135 KVUE
START  = "2023-09-01"   # KVUE listed Sep 2023
END    = "2024-12-31"
INITIAL_VALUE = 10_000

# Asymmetric book sizes derived from hedge ratio
LONG_BS  = 1.0 / (1.0 + RATIO)   # ≈ 0.468
SHORT_BS = RATIO / (1.0 + RATIO)  # ≈ 0.532
print(f"long_book_size={LONG_BS:.4f}  short_book_size={SHORT_BS:.4f}  ratio={SHORT_BS/LONG_BS:.4f}")

long_book_size=0.4684  short_book_size=0.5316  ratio=1.1350


In [2]:
yf = YFinance(auto_adjust=True)
df = yf.query([LONG, SHORT, CASH], start_date=START, end_date=END)

prices = {}
for symbol in df["symbol"].unique():
    sub = df[df["symbol"] == symbol].set_index("date")[["open", "high", "low", "close"]]
    prices[symbol] = sub

print(f"Loaded {len(prices)} symbols")
for sym, d in prices.items():
    print(f"  {sym}: {len(d)} rows  {d.index[0].date()} → {d.index[-1].date()}")

Loaded cached bar data.

Loaded 3 symbols
  KVUE: 334 rows  2023-09-01 → 2024-12-30
  TXN: 334 rows  2023-09-01 → 2024-12-30
  BIL: 334 rows  2023-09-01 → 2024-12-30


In [3]:
strategy = DollarNeutral(
    long_weights={LONG: 1.0},
    short_weights={SHORT: 1.0},
    cash_symbol=CASH,
    long_book_size=LONG_BS,
    short_book_size=SHORT_BS,
    tolerance=0.05,
)

engine = ScheduleBasedEngine(
    allocation=strategy,
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)

result = engine.run(
    symbols=[LONG, SHORT, CASH],
    start=START, end=END,
    prices_df=prices,
)
print(result.summary())

Backtest Summary
----------------
Sharpe Ratio:        0.5129
Sortino Ratio:       0.7282
MAR Ratio:           0.7888
CAGR:                11.26%
Max Drawdown:        14.28%
Kelly Leverage:      3.3120
Mean Excess Return:  0.0794
Final Value:         11,525.64
Total Fee:           0.65
Rebalances:          16


In [4]:
fig = result.plot_portfolio(mark_rebalances=True)
fig.show()

In [5]:
decisions_df = rebalance_decisions_table(result)
decisions_df

,date,equity_before,equity_after,fee_paid,TXN_price,TXN_qty_before,TXN_trade_qty,TXN_qty_after,TXN_value_after,KVUE_price,KVUE_qty_before,KVUE_trade_qty,KVUE_qty_after,KVUE_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after
0,2023-09-15 00:00:00+00:00,10223.789,10223.789,0.000,151.752,29.555,0.000,29.555,4484.992,19.170,-256.297,0.000,-256.297,-4913.281,81.689,130.398,0.000,130.398,10652.078
1,2023-10-16 00:00:00+00:00,10428.053,10428.053,0.000,143.988,29.555,0.000,29.555,4255.530,17.653,-256.297,0.000,-256.297,-4524.292,82.032,130.398,0.000,130.398,10696.816
2,2023-11-15 00:00:00+00:00,10384.708,10384.708,0.000,143.019,29.555,0.000,29.555,4226.905,17.892,-256.297,0.000,-256.297,-4585.582,82.389,130.398,0.000,130.398,10743.385
3,2023-12-15 00:00:00+00:00,10576.671,10576.671,0.000,158.812,29.555,0.000,29.555,4693.654,19.161,-256.297,0.000,-256.297,-4910.784,82.776,130.398,0.000,130.398,10793.800
4,2024-01-16 00:00:00+00:00,10433.131,10433.131,0.000,154.330,29.555,0.000,29.555,4561.172,19.380,-256.297,0.000,-256.297,-4966.933,83.121,130.398,0.000,130.398,10838.891
5,2024-02-15 00:00:00+00:00,10894.792,10894.792,0.000,152.539,29.555,0.000,29.555,4508.240,17.569,-256.297,0.000,-256.297,-4502.876,83.509,130.398,0.000,130.398,10889.427
6,2024-03-15 00:00:00+00:00,10965.839,10965.839,0.000,163.748,29.555,0.000,29.555,4839.535,18.759,-256.297,0.000,-256.297,-4807.794,83.851,130.398,0.000,130.398,10934.098
7,2024-04-15 00:00:00+00:00,11123.626,11123.626,0.000,157.892,29.555,0.000,29.555,4666.454,17.643,-256.297,0.000,-256.297,-4521.785,84.195,130.398,0.000,130.398,10978.957
8,2024-05-15 00:00:00+00:00,11643.608,11643.316,0.292,186.926,29.555,-0.379,29.176,5453.680,19.144,-256.297,-67.041,-323.338,-6189.927,84.553,130.398,16.017,146.416,12379.855
9,2024-06-14 00:00:00+00:00,12400.064,12400.064,0.000,185.367,29.176,0.000,29.176,5408.216,16.842,-323.338,0.000,-323.338,-5445.569,84.946,146.416,0.000,146.416,12437.417


## Comparison to Baselines

We compare the dollar-neutral spread against three single-leg alternatives:
- **Long TXN**: 100% TXN, buy-and-hold the tech analog
- **Short KVUE** (synthetic): 100% BIL + KVUE at −100% — pure short consumer-staples exposure
- **50/50 TXN+BIL**: half in TXN, half in T-bills — a simple risk-managed long


In [6]:
# Baseline 1: 100% long TXN
engine_txn = ScheduleBasedEngine(
    allocation=FixRatio(weights={LONG: 1.0}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_long_txn = engine_txn.run(
    symbols=[LONG], start=START, end=END, prices_df={LONG: prices[LONG]}
)
print("Long TXN only")
print(result_long_txn.summary())

Long TXN only
Backtest Summary
----------------
Sharpe Ratio:        0.3792
Sortino Ratio:       0.6205
MAR Ratio:           0.6651
CAGR:                10.98%
Max Drawdown:        16.51%
Kelly Leverage:      1.4589
Mean Excess Return:  0.0986
Final Value:         11,487.21
Total Fee:           0.00
Rebalances:          0


In [7]:
# Baseline 2: 50% TXN / 50% BIL
engine_half = ScheduleBasedEngine(
    allocation=FixRatio(weights={LONG: 0.5, CASH: 0.5}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_half = engine_half.run(
    symbols=[LONG, CASH], start=START, end=END,
    prices_df={LONG: prices[LONG], CASH: prices[CASH]},
)
print("50% TXN / 50% BIL")
print(result_half.summary())

50% TXN / 50% BIL
Backtest Summary
----------------
Sharpe Ratio:        0.3976
Sortino Ratio:       0.6475
MAR Ratio:           1.0344
CAGR:                8.62%
Max Drawdown:        8.34%
Kelly Leverage:      3.0555
Mean Excess Return:  0.0517
Final Value:         11,163.49
Total Fee:           0.11
Rebalances:          16


In [8]:
compare_strategies(
    result, result_long_txn, result_half,
    names=["DollarNeutral TXN/KVUE", "Long TXN", "50/50 TXN+BIL"],
)

,DollarNeutral TXN/KVUE,Long TXN,50/50 TXN+BIL,better
metric,,,,
sharpe_ratio,0.512945,0.379199,0.397596,DollarNeutral TXN/KVUE
sortino_ratio,0.728165,0.620461,0.647509,DollarNeutral TXN/KVUE
mar_ratio,0.788809,0.665116,1.034387,50/50 TXN+BIL
cagr,0.112613,0.109823,0.086235,DollarNeutral TXN/KVUE
max_drawdown,0.142763,0.165119,0.083368,50/50 TXN+BIL


In [9]:
plot_strategy_comparison_interactive(
    result, result_long_txn, result_half,
)

## Portfolio Beta

Track portfolio beta over time vs SPY.


In [10]:
# Plot portfolio beta over time
fig_beta = result.plot_portfolio_beta()
fig_beta.show()


Loaded cached bar data.



## Interpretation

### Hedge Ratio 1:1.135
The ratio of 1.135 is derived from the OLS regression of TXN returns on KVUE returns over a calibration
period. A \$1 long position in TXN is matched with \$1.135 short in KVUE so that
sector/market moves roughly cancel, leaving the **spread return** (TXN outperformance vs KVUE) as the
primary P&L driver.

### Why DollarNeutral?
Unlike a pure long position in TXN, the spread strategy:
- Reduces market/sector beta — TXN and KVUE are both large-cap consumer/tech; correlated macro moves offset
- Profits when TXN outperforms KVUE **regardless of overall market direction**
- The 5% tolerance band avoids unnecessary trading on small drifts

### BIL collateral
Short positions require posted margin. BIL represents the T-bill collateral held against the KVUE short,
earning near risk-free yield while the position is open.

> **⚠ Margin required**: Short selling requires a margin/prime brokerage account.
